# Carnet de démarrage — sujet 05 · Santander Satisfaction

**Objectif du sujet** : repérer le client qui va se plaindre.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

3,96 % de mécontents. 97 colonnes sont constantes sur un échantillon.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **(lien du Drive communiqué par l’enseignant)** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_05_Santander_Satisfaction/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |

**Format de sortie attendu** : `ID,TARGET`.
**Colonne d'identifiant** : `ID`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


In [5]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"
os.environ["DONNEES_PROJET"] = str(Path.cwd().parent / "data")


In [6]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv']
SUJET = "Sujet_05_Santander_Satisfaction"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : /home/vino-yd/santander-satisfaction/santander-satisfaction/data
fichiers vus : ['.gitkeep', 'a_decider.csv', 'train.csv']


In [7]:
# --- 3. Chargement (table légère) ---------------------------------------------
def table(nom, colonnes=None, morceaux=None, complet=False):
    """Ouvre un CSV directement avec pandas — la table tient en mémoire."""
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")
    if morceaux:
        return pd.read_csv(source, usecols=colonnes, chunksize=morceaux)
    return pd.read_csv(source, usecols=colonnes, nrows=None if complet else ECHANTILLON)


In [8]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (60816, 371)
colonnes : 371


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,130807,2,26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0
1,78981,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85472.340000,0
2,50325,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,317769.240000,0
3,146646,2,30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76209.960000,0
4,136536,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,302754.000000,0


In [9]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    60816.000000
mean         0.039562
std          0.194929
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: TARGET, dtype: float64
TARGET
0    0.960438
1    0.039562
Name: proportion, dtype: float64


In [10]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['ID', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"ID": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["ID"], complet=True)
identifiants = a_predire["ID"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 15204
écrit : soumission.csv · 15,204 lignes · colonnes ['ID', 'TARGET']


,ID,TARGET
0,38893,None
1,133597,None
2,24908,None
3,19551,None
4,36051,None
...,...,...
15199,28686,None
15200,66259,None
15201,44895,None
15202,33938,None


In [11]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
raise NotImplementedError(
    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
)


NotImplementedError: Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici.

In [13]:
# ============================================================
# P1 — 1.1 : SYNTHESE DE L'INVENTAIRE
# ============================================================

resume = pd.DataFrame({
    "Indicateurs": [
        "Nombre de lignes",
        "Nombre de colonnes",
        "Valeurs manquantes",
        "Lignes dupliquées",
        "Identifiants distincts",
        "Identifiants manquants",
        "Mémoire utilisée (Mo)"
    ],
    "TRAIN": [
        len(train_complet),
        train_complet.shape[1],
        int(train_complet.isna().sum().sum()),
        int(train_complet.duplicated().sum()),
        train_complet["ID"].nunique(),
        int(train_complet["ID"].isna().sum()),
        round(train_complet.memory_usage(deep=True).sum() / 1024**2, 2)
    ],
    "A_DECIDER": [
        len(test_complet),
        test_complet.shape[1],
        int(test_complet.isna().sum().sum()),
        int(test_complet.duplicated().sum()),
        test_complet["ID"].nunique(),
        int(test_complet["ID"].isna().sum()),
        round(test_complet.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

display(resume)

print("\n=== STRUCTURE DES TABLES ===")
print("Colonnes communes :", len(
    (set(train_complet.columns) - {"TARGET"})
    & set(test_complet.columns)
))
print("TARGET présente dans TRAIN :", "TARGET" in train_complet.columns)
print("TARGET présente dans A_DECIDER :", "TARGET" in test_complet.columns)

print("\n=== TYPES DE DONNEES DE TRAIN ===")
display(train_complet.dtypes.value_counts().rename_axis("Type").to_frame("Nombre"))

print("\n=== REPARTITION DE TARGET ===")
display(train_complet["TARGET"].value_counts(normalize=True).mul(100).round(2).rename("Pourcentage"))

,Indicateurs,TRAIN,A_DECIDER
0,Nombre de lignes,60816.00,15204.00
1,Nombre de colonnes,371.00,370.00
2,Valeurs manquantes,0.00,0.00
3,Lignes dupliquées,0.00,0.00
4,Identifiants distincts,60816.00,15204.00
5,Identifiants manquants,0.00,0.00
6,Mémoire utilisée (Mo),172.14,42.92



=== STRUCTURE DES TABLES ===
Colonnes communes : 370
TARGET présente dans TRAIN : True
TARGET présente dans A_DECIDER : False

=== TYPES DE DONNEES DE TRAIN ===


,Nombre
Type,
int64,260
float64,111



=== REPARTITION DE TARGET ===


TARGET
0    96.04
1     3.96
Name: Pourcentage, dtype: float64

In [14]:

# ============================================================
# P1.8 — DÉTECTION DES FUITES DE DONNÉES
# ============================================================

import pandas as pd
import numpy as np

# 1. Charger les données si elles ne sont pas déjà chargées
if "train_complet" not in globals():
    train_complet = table("train.csv", complet=True)

if "test_complet" not in globals():
    test_complet = table("a_decider.csv", complet=True)

CIBLE = "TARGET"

if CIBLE not in train_complet.columns:
    raise ValueError("La colonne TARGET est absente de train.csv")

# 2. Préparer les variables
y = train_complet[CIBLE]
anomalies = []

# 3. Vérifier les colonnes dont le nom peut être suspect
mots_suspects = [
    "target", "label", "outcome", "result",
    "response", "fraud", "satisfaction"
]

for colonne in train_complet.columns:
    if colonne == CIBLE:
        continue

    nom = str(colonne).lower()

    if any(mot in nom for mot in mots_suspects):
        anomalies.append({
            "colonne": colonne,
            "indice": "Nom potentiellement lié à la cible",
            "observation": "À examiner manuellement"
        })

# 4. Rechercher une copie directe ou inversée de TARGET
for colonne in train_complet.columns:
    if colonne in [CIBLE, "ID"]:
        continue

    serie = train_complet[colonne]

    if serie.isna().any() or y.isna().any():
        continue

    if serie.equals(y):
        anomalies.append({
            "colonne": colonne,
            "indice": "Copie exacte de TARGET",
            "observation": "Fuite de données très probable"
        })

    elif pd.api.types.is_numeric_dtype(serie):
        if serie.equals(1 - y):
            anomalies.append({
                "colonne": colonne,
                "indice": "Copie inversée de TARGET",
                "observation": "Fuite de données très probable"
            })

# 5. Rechercher les corrélations numériques très élevées
for colonne in train_complet.select_dtypes(include="number").columns:
    if colonne in [CIBLE, "ID"]:
        continue

    correlation = train_complet[colonne].corr(y)

    if pd.notna(correlation) and abs(correlation) >= 0.98:
        anomalies.append({
            "colonne": colonne,
            "indice": f"Corrélation très élevée : {correlation:.4f}",
            "observation": "Vérifier un possible lien direct avec TARGET"
        })

# 6. Rechercher les colonnes qui séparent presque parfaitement les classes
for colonne in train_complet.columns:
    if colonne in [CIBLE, "ID"]:
        continue

    serie = train_complet[colonne]

    # On évite les colonnes ayant trop de valeurs différentes
    if serie.nunique(dropna=False) > 50:
        continue

    groupes = (
        pd.DataFrame({colonne: serie, CIBLE: y})
        .groupby(colonne, dropna=False)[CIBLE]
        .agg(["count", "mean"])
    )

    groupes = groupes[groupes["count"] >= 20]

    if groupes.empty:
        continue

    purete_max = np.maximum(
        groupes["mean"],
        1 - groupes["mean"]
    ).max()

    if purete_max >= 0.999:
        anomalies.append({
            "colonne": colonne,
            "indice": f"Classe presque parfaitement identifiable ({purete_max:.3f})",
            "observation": "Examiner cette variable pour une fuite possible"
        })

# 7. Vérifier les différences de colonnes entre train et test
colonnes_train = set(train_complet.columns) - {CIBLE}
colonnes_test = set(test_complet.columns)

absentes_test = sorted(colonnes_train - colonnes_test)
absentes_train = sorted(colonnes_test - colonnes_train)

# 8. Afficher les résultats de façon synthétique
print("========== P1.8 : DÉTECTION DES FUITES ==========")

print("\n1. Nombre de colonnes examinées :",
      len(train_complet.columns) - 1)

print("\n2. Colonnes absentes du fichier test :",
      absentes_test)

print("\n3. Colonnes présentes uniquement dans le fichier test :",
      absentes_train)

print("\n4. Nombre d'indices de fuite détectés :",
      len(anomalies))

if anomalies:
    resultat_fuites = pd.DataFrame(anomalies)
    display(resultat_fuites)
else:
    print("Aucun indice suspect détecté par ces contrôles.")

print("\n========== FIN DU P1.8 ==========")


========== P1.8 : DÉTECTION DES FUITES ==========

1. Nombre de colonnes examinées : 370

2. Colonnes absentes du fichier test : []

3. Colonnes présentes uniquement dans le fichier test : []

4. Nombre d'indices de fuite détectés : 38


,colonne,indice,observation
0,ind_var20_0,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
1,ind_var20,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
2,ind_var33_0,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
3,ind_var33,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
4,num_var4,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
5,num_var12,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
6,num_var13_0,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
7,num_var13_corto_0,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
8,num_var13_largo_0,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible
9,num_var13_largo,Classe presque parfaitement identifiable (1.000),Examiner cette variable pour une fuite possible



========== FIN DU P1.8 ==========
